# Supplementary Figure 1: Binary Empathic Response Learning Curves for Supervised Models

This notebook generates Supplementary Figure 1. It uses only aggregate F1 scores and confidence intervals and contains no patient-level data or PHI.

In [ ]:
# Supplementary Figure 1: Binary Empathic Response Learning Curves for Supervised Models
# This notebook generates the accessibility-focused Supplementary Figure 1.
# It uses only aggregate F1 scores and confidence intervals and contains no patient-level data or PHI.

import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
from pathlib import Path

training_sizes = np.array([10, 25, 50, 100, 150, 200])

deberta_f1 = np.array([0.0625, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000])
deberta_low = np.array([0.0101, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000])
deberta_high = np.array([0.1150, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000])

clinicalbert_f1 = np.array([0.0325, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000])
clinicalbert_low = np.array([0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000])
clinicalbert_high = np.array([0.0695, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000])

llama_best = 0.79

def yerr_from_bounds(f1, low, high):
    return np.vstack([f1 - low, high - f1])

for model, f1, low, high in [
    ("DeBERTaV3-base", deberta_f1, deberta_low, deberta_high),
    ("ClinicalBERT", clinicalbert_f1, clinicalbert_low, clinicalbert_high)
]:
    for n, f, lo, hi in zip(training_sizes, f1, low, high):
        assert lo <= f <= hi, (model, n, f, lo, hi)

available_fonts = {f.name for f in font_manager.fontManager.ttflist}
font_name = "Open Sans" if "Open Sans" in available_fonts else "DejaVu Sans"

plt.rcParams.update({
    "font.family": font_name,
    "font.size": 11,
    "axes.labelsize": 12,
    "xtick.labelsize": 10.5,
    "ytick.labelsize": 10.5,
    "legend.fontsize": 10.5,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

deberta_color = "#0072B2"
clinicalbert_color = "#D55E00"
llama_color = "#111111"
grid_color = "#D0D0D0"
spine_color = "#777777"
text_color = "#111111"

fig, ax = plt.subplots(figsize=(8.6, 5.8))

ax.errorbar(
    training_sizes, deberta_f1,
    yerr=yerr_from_bounds(deberta_f1, deberta_low, deberta_high),
    fmt="none", ecolor=deberta_color, elinewidth=1.5,
    capsize=0, alpha=0.9, zorder=1
)
ax.errorbar(
    training_sizes, clinicalbert_f1,
    yerr=yerr_from_bounds(clinicalbert_f1, clinicalbert_low, clinicalbert_high),
    fmt="none", ecolor=clinicalbert_color, elinewidth=1.5,
    capsize=0, alpha=0.9, zorder=1
)

ax.plot(
    training_sizes, deberta_f1,
    color=deberta_color, linestyle="-", linewidth=2.1,
    marker="o", markersize=6.5,
    markerfacecolor=deberta_color,
    markeredgecolor="white", markeredgewidth=0.8,
    label="DeBERTaV3-base", zorder=3
)
ax.plot(
    training_sizes, clinicalbert_f1,
    color=clinicalbert_color, linestyle=(0, (5, 2)), linewidth=2.1,
    marker="s", markersize=6.5,
    markerfacecolor=clinicalbert_color,
    markeredgecolor="white", markeredgewidth=0.8,
    label="ClinicalBERT", zorder=3
)

ax.axhline(
    llama_best, color=llama_color, linestyle=(0, (2, 2)),
    linewidth=1.9, label="Llama-3.1-70B", zorder=2
)

ax.set_xlim(0, 210)
ax.set_ylim(0, 1.0)
ax.set_xticks(training_sizes)
ax.set_yticks(np.arange(0, 1.01, 0.2))
ax.set_xlabel("Training set size")
ax.set_ylabel("F1 score")
ax.set_title("Empathy", fontsize=13.5, fontweight="semibold", pad=12)

ax.yaxis.grid(True, color=grid_color, linewidth=0.8, linestyle="--", dashes=(2, 2), alpha=0.9)
ax.xaxis.grid(False)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
for spine in ["left", "bottom"]:
    ax.spines[spine].set_color(spine_color)
    ax.spines[spine].set_linewidth(0.9)
ax.tick_params(colors=text_color)

ax.legend(
    loc="upper right",
    frameon=True,
    framealpha=1,
    edgecolor="#DDDDDD",
    facecolor="white",
    borderpad=0.7,
    handlelength=2.8
)

fig.tight_layout()

outdir = Path("figures")
outdir.mkdir(exist_ok=True)
fig.savefig(outdir / "Supplementary_Figure1.png", dpi=600, bbox_inches="tight", facecolor="white")
fig.savefig(outdir / "Supplementary_Figure1.pdf", bbox_inches="tight", facecolor="white")
fig.savefig(outdir / "Supplementary_Figure1.svg", bbox_inches="tight", facecolor="white")
plt.close(fig)

print(f"Font used: {font_name}")
